# WFS Diffraction: does it bias the danish wavefront fit?

**Author:** Aaron Roodman   **Created:** 2026-06-17   **Status:** in progress

**Motivation.** danish fits donuts in the **geometric (photon-ray) approximation**, ignoring
diffraction. Three RubinTV data observations (WFS and FAM): the spherical **Z11 differs ~0.3 um**
between **separately**-fit intra and extra donuts; rim residuals (esp. intra); struts sharper
extra. This notebook tests whether **diffraction** drives it, on-axis, by generating diffraction
donuts and fitting them with danish, **paired** and **unpaired**.

**Generator.** Manual **Fraunhofer FFT** on an explicit annular pupil (N=4096; 2048 vs 4096
convergence checked below): `donut = |FFT(aperture * exp(2*pi*i*W/lambda))|^2`, `W` the off-axis
annular-Zernike wavefront from the ts_wep `Instrument`. Validated against the geometric ray donut.
Struts omitted (LSST's double-bladed spider is not easily modelled). The seeing kernel is
**Kolmogorov, 0.7 arcsec FWHM** — matching what the fit assumes.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-17 | Aaron Roodman | Initial: validated Fraunhofer-FFT diffraction donut + production danish fit; diffraction gives a Z11 intra/extra split in the unpaired fits, hidden in the paired fit. |
| 2026-06-17 | Aaron Roodman | Kolmogorov 0.7" seeing (matches the fit); geometric-vs-FFT difference maps + radial/fractional profiles; 2048-vs-4096 FFT-grid convergence test; per-side fwhm in the fit table; computed (non-hardcoded) summary; default grid N=4096. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Generate & Validate the Donuts](#donuts)
5. [Geometric vs FFT-Diffraction (extra): difference maps & profiles](#geomvsfft)
6. [Intra vs Extra: spherical breaks the symmetry](#intraextra)
7. [FFT-grid convergence: 2048 vs 4096](#gridconv)
8. [Danish Fits: Z11 intra/extra split (paired & unpaired)](#fits)
9. [Data / Model / Residual](#resid)
10. [Summary](#summary)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================
detector      = "R30_S21"     # science det -> FAM (camera-hexapod) instrument; FoV-centre fit (theta=0)
band          = "i"
wl_nm         = 754.0         # i-band effective wavelength (nm)
defocal_mm    = 1.5           # FAM camera-hexapod defocus
seeing_arcsec = 0.7           # Kolmogorov seeing FWHM (matches the danish fit assumption)
npix          = 185           # detector stamp (pixels), 0.2"/pix
peak_e        = 1.0e4         # donut peak electrons/pixel
sky_e         = 200.0         # sky pedestal (e/pix)
# Production danish fit config (FAM-style, FoV centre):
noll          = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26]  # standard 21
fwhm_bounds   = (0.5, 1.5)    # arcsec; contains the danish x0 (1.0), caps the runaway
# Fraunhofer-FFT pupil grid:
Ngrid         = 4096          # pupil-plane grid for the analysis
grids_compare = [2048, 4096]  # for the grid-convergence test
gridpad       = 1.15          # pupil half-width = gridpad * R_outer
n_band        = 9             # broadband wavelength samples across i-band
cmap          = "inferno"

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import batoid
import danish
import galsim
import galsim.zernike as gz
from scipy.ndimage import gaussian_filter, zoom
from scipy.signal import fftconvolve
from lsst.ts.wep.utils import getTaskInstrument, DefocalType, BandLabel
from lsst.ts.wep.image import Image
from lsst.ts.wep.estimation import WfEstimator

inst = getTaskInstrument("LSSTCam", detector)
inst.defocalOffset = defocal_mm * 1e-3
fid = inst.getBatoidModel(band)
wl = wl_nm * 1e-9
R = inst.radius; Rin = R * inst.obscuration
pix = inst.pixelSize; fl = inst.focalLength
detplate = np.degrees(pix / fl) * 3600          # detector arcsec/pixel (0.2)
print(f"R_outer={R:.3f} m, obscuration={inst.obscuration:.3f}, detector {detplate:.3f} arcsec/pix")
print(f"on-axis design Z11 (spherical) = "
      f"{1e6*inst.getOffAxisCoeff(0,0,DefocalType.Extra,band,nollIndicesModel=np.arange(79),nollIndicesIntr=np.arange(4,23))[11]:.3f} um")

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
# ---- Kolmogorov seeing kernel (matches the danish atmosphere assumption) ----
def kolmogorov_kernel(plate_as, fwhm=None):
    """Normalised Kolmogorov PSF kernel sampled at plate_as arcsec/pixel."""
    fwhm = seeing_arcsec if fwhm is None else fwhm
    nk = int(8 * fwhm / plate_as) | 1                       # odd, ~8 FWHM wide (wings)
    k = galsim.Kolmogorov(fwhm=fwhm).drawImage(nx=nk, ny=nk, scale=plate_as, method="no_pixel").array
    return k / k.sum()


def convolve_seeing(img, plate_as):
    return fftconvolve(img, kolmogorov_kernel(plate_as), mode="same")


# ---- pupil grid (cached per N) ----
_pupil = {}
def get_pupil(N):
    if N not in _pupil:
        u = np.linspace(-R * gridpad, R * gridpad, N)
        U, V = np.meshgrid(u, u)
        _pupil[N] = (U, V, (np.hypot(U, V) >= Rin) & (np.hypot(U, V) <= R), u[1] - u[0])
    return _pupil[N]


def fft_donut(dtype, wl_, N):
    """Monochromatic Fraunhofer diffraction donut on the N pupil grid (off-axis annular-Zernike
    wavefront for this defocal side). Returns (donut, arcsec/FFT-pixel)."""
    U, V, ap, du = get_pupil(N)
    ab = inst.getOffAxisCoeff(0, 0, dtype, band, nollIndicesModel=np.arange(79), nollIndicesIntr=np.arange(4, 23))
    W = np.where(ap, gz.Zernike(ab, R_outer=R, R_inner=Rin)(U, V), 0.0)
    field = ap * np.exp(2j * np.pi * W / wl_)
    donut = np.abs(np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(field)))) ** 2
    return donut, np.degrees(wl_ / (N * du)) * 3600


def to_detector(donut, fft_plate):
    """Kolmogorov-seeing-convolve at the FINE FFT scale (smooths the rings before downsampling),
    then downsample to the detector grid and centre in an npix stamp."""
    s = convolve_seeing(np.clip(donut, 0, None), fft_plate)
    z = zoom(s, fft_plate / detplate)
    n = z.shape[0]; out = np.zeros((npix, npix))
    if n <= npix:
        o = (npix - n) // 2; out[o:o + n, o:o + n] = z
    else:
        c = n // 2; h = npix // 2; out = z[c - h:c - h + npix, c - h:c - h + npix]
    return out


def diffraction_donut(dtype, seed, N=None, broadband=True, noise=True):
    """Detector-grid diffraction donut. Broadband sums over the i-band so the monochromatic
    Fresnel rings wash out, as in real data."""
    N = Ngrid if N is None else N
    if broadband:
        wls = np.linspace(0.692, 0.818, n_band) * 1e-6
        wts = np.exp(-0.5 * ((wls - wl) / 0.05e-6) ** 2); wts /= wts.sum()
    else:
        wls, wts = [wl], [1.0]
    acc = np.zeros((npix, npix))
    for wl_, wt in zip(wls, wts):
        d, pl = fft_donut(dtype, wl_, N); acc += wt * to_detector(d, pl)
    acc = acc * peak_e / acc.max()
    if not noise:
        return acc
    return np.random.default_rng(seed).poisson(acc + sky_e).astype(float)


def geom_donut(dtype, seed, osamp=4, noise=True):
    """Geometric (photon-ray) donut: histogram of un-vignetted ray landings, Kolmogorov-blurred."""
    sign = +1 if dtype == DefocalType.Extra else -1
    tel = fid.withLocallyShiftedOptic("LSSTCamera", [0, 0, sign * defocal_mm * 1e-3])
    r = batoid.RayVector.asPolar(optic=tel, wavelength=wl, theta_x=0, theta_y=0, nrad=300, naz=1800, inner=0)
    tel.trace(r); ok = ~r.vignetted & ~r.failed
    x, y = r.x[ok], r.y[ok]; cx, cy = x.mean(), y.mean()
    half = npix * pix / 2; nb = npix * osamp
    H, _, _ = np.histogram2d(x, y, bins=nb, range=[[cx - half, cx + half], [cy - half, cy + half]])
    H = convolve_seeing(H.T, detplate / osamp)
    img = H.reshape(npix, osamp, npix, osamp).sum(axis=(1, 3))
    img = img * peak_e / img.max()
    if not noise:
        return img
    return np.random.default_rng(seed).poisson(img + sky_e).astype(float)


# ---- danish fit (production config) ----
_w0 = WfEstimator(algoName="danish", algoConfig={"jointFitPair": False}, instConfig=inst, nollIndices=np.array(noll))
_zr = inst.getOffAxisCoeff(0, 0, DefocalType.Extra, band, nollIndicesModel=np.arange(79), nollIndicesIntr=np.arange(4, 23))
_fac = danish.DonutFactory(R_outer=R, R_inner=Rin, mask_params=inst.maskParams, focal_length=fl, pixel_scale=pix)
_m = danish.SingleDonutModel(_fac, z_ref=_zr, z_terms=tuple(noll), thx=0, thy=0, npix=npix, bkg_order=_w0.algo.bkgOrder)
_b = _m.pack_params(flux=[-np.inf, np.inf], dx=[-np.inf, np.inf], dy=[-np.inf, np.inf],
                    fwhm=list(fwhm_bounds), bkg=[[-np.inf, np.inf]] * _m.nbkg, z_fit=[[-np.inf, np.inf]] * len(noll))
SINGLE_BOUNDS = tuple([list(x) for x in zip(*_b)])


def single_fit(img, dtype):
    wfe = WfEstimator(algoName="danish",
                      algoConfig={"jointFitPair": False, "binning": 1, "lstsqKwargs": {"bounds": SINGLE_BOUNDS, "x_scale": "jac"}},
                      instConfig=inst, nollIndices=np.array(noll), startWithIntrinsic=True, units="um", saveHistory=True)
    zk, meta = wfe.estimateZk(Image(img, (0.0, 0.0), dtype, BandLabel.LSST_I), None)
    return np.asarray(zk).ravel(), float(meta.get("fwhm", np.nan)), wfe.history


def paired_fit(imgE, imgI):
    wfe = WfEstimator(algoName="danish", algoConfig={"jointFitPair": True, "binning": 1, "lstsqKwargs": {"x_scale": "jac"}},
                      instConfig=inst, nollIndices=np.array(noll), startWithIntrinsic=True, units="um", saveHistory=True)
    zk, meta = wfe.estimateZk(Image(imgE, (0.0, 0.0), DefocalType.Extra, BandLabel.LSST_I),
                              Image(imgI, (0.0, 0.0), DefocalType.Intra, BandLabel.LSST_I))
    return np.asarray(zk).ravel(), float(meta.get("fwhm", np.nan)), wfe.history


i11 = noll.index(11)

def radial_profile(img, nbin=70, rmax=None):
    n = img.shape[0]; yy, xx = np.mgrid[0:n, 0:n]; c = (n - 1) / 2; rr = np.hypot(xx - c, yy - c)
    rmax = n / 2 if rmax is None else rmax
    e = np.linspace(0, rmax, nbin + 1); rc = 0.5 * (e[:-1] + e[1:])
    p = np.array([img[(rr >= e[k]) & (rr < e[k + 1])].mean() if ((rr >= e[k]) & (rr < e[k + 1])).any() else np.nan for k in range(nbin)])
    return rc, p

<a id='donuts'></a>
## 4. Generate & Validate the Donuts

Geometric (ray) and Fraunhofer-FFT (diffraction) donuts, extra and intra, on the detector grid, with Kolmogorov 0.7 arcsec seeing. The FFT donut should match the geometric one in size with a clean central hole.

In [ ]:
donuts = {
    "geometric": {"extra": geom_donut(DefocalType.Extra, 1), "intra": geom_donut(DefocalType.Intra, 2)},
    "FFT diffraction": {"extra": diffraction_donut(DefocalType.Extra, 1), "intra": diffraction_donut(DefocalType.Intra, 2)},
}
fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
for col, kind in enumerate(["geometric", "FFT diffraction"]):
    for row, side in enumerate(["extra", "intra"]):
        axes[row, col].imshow(donuts[kind][side], origin="lower", cmap=cmap)
        axes[row, col].set_title(f"{kind} - {side}", fontsize=10); axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
fig.suptitle(f"On-axis donuts (FAM +/-{defocal_mm} mm, {band}-band, Kolmogorov {seeing_arcsec}\" seeing, N={Ngrid})", fontsize=11)
plt.show()
yy, xx = np.mgrid[0:npix, 0:npix]; c = (npix - 1) / 2; rr = np.hypot(xx - c, yy - c)
for kind in ["geometric", "FFT diffraction"]:
    a = donuts[kind]["extra"] - sky_e; a = a / a.max()
    print(f"{kind:16s}: central(r<25)={a[rr<25].mean():.3f}  outer half-max r={rr[a>0.5].max():.0f} px")

<a id='geomvsfft'></a>
## 5. Geometric vs FFT-Diffraction (extra focal): difference & profiles

Direct comparison of the extra-focal geometric and FFT (diffraction) donuts (each unit-sum normalised): the **difference map** (FFT − geometric) shows where diffraction reshapes the donut; the **radial profiles** and their **fractional difference** quantify it. The diffraction structure is concentrated at the inner/outer rim.

In [ ]:
g = donuts["geometric"]["extra"] - sky_e; f = donuts["FFT diffraction"]["extra"] - sky_e
g = g / g.sum(); f = f / f.sum()                       # unit-sum normalise (compare shape, not flux)
diff = f - g
rc, pg = radial_profile(g); _, pf = radial_profile(f)
frac = np.where(pg > 0.02 * np.nanmax(pg), (pf - pg) / pg, np.nan)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
v0 = max(g.max(), f.max())
ax[0].imshow(f, origin="lower", cmap=cmap, vmin=0, vmax=v0); ax[0].set_title("FFT diffraction (extra)"); ax[0].set_xticks([]); ax[0].set_yticks([])
vv = np.nanpercentile(np.abs(diff), 99.5)
im = ax[1].imshow(diff, origin="lower", cmap="RdBu_r", vmin=-vv, vmax=vv); plt.colorbar(im, ax=ax[1], fraction=0.046)
ax[1].set_title("FFT - geometric (difference map)"); ax[1].set_xticks([]); ax[1].set_yticks([])
ax2b = ax[2].twinx()
ax[2].plot(rc, pg / np.nanmax(pg), label="geometric", color="C0")
ax[2].plot(rc, pf / np.nanmax(pf), label="FFT diffraction", color="C3")
ax2b.plot(rc, 100 * frac, color="k", lw=1, ls="--", label="fractional diff")
ax[2].set_xlabel("radius [pix]"); ax[2].set_ylabel("norm azimuthal mean"); ax2b.set_ylabel("(FFT-geom)/geom [%]")
ax[2].legend(loc="upper left", fontsize=8); ax2b.legend(loc="upper right", fontsize=8); ax[2].set_title("radial profiles + fractional diff")
fig.suptitle("Extra-focal: geometric vs FFT-diffraction donut", fontsize=12)
plt.show()
print(f"extra donut: RMS(FFT-geom) = {np.sqrt(np.mean(diff**2)):.2e} (unit-sum); "
      f"peak |fractional diff| in annulus = {np.nanmax(np.abs(frac))*100:.1f}%")

<a id='intraextra'></a>
## 6. Intra vs Extra: spherical breaks the symmetry

For *pure* defocus the Fraunhofer intra and extra donuts are identical (conjugate symmetry); the design spherical aberration makes them differ, concentrated at the rim (matching the data's rim residuals). Full-resolution FFT donuts, no seeing.

In [ ]:
E2, plE = fft_donut(DefocalType.Extra, wl, Ngrid); I2, plI = fft_donut(DefocalType.Intra, wl, Ngrid)
E2 = E2 / E2.max(); I2 = I2 / I2.max()
cc = Ngrid // 2; ww = min(int(15 / plE), cc - 1)
yy2, xx2 = np.mgrid[0:Ngrid, 0:Ngrid]; rr2 = np.hypot(xx2 - cc, yy2 - cc)
nb = 90; e = np.linspace(0, 16, nb + 1) / plE; rax = 0.5 * (e[:-1] + e[1:]) * plE
pE = np.array([E2[(rr2 >= e[k]) & (rr2 < e[k + 1])].mean() for k in range(nb)])
pI = np.array([I2[(rr2 >= e[k]) & (rr2 < e[k + 1])].mean() for k in range(nb)])
print(f"extra-intra donut RMS diff = {np.sqrt(np.mean((E2-I2)**2)):.4f} (peak-normed)")
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
ext = ww * plE; dimg = (E2 - I2)[cc - ww:cc + ww, cc - ww:cc + ww]; v = np.percentile(np.abs(dimg), 99.5)
im = ax[0].imshow(dimg, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=[-ext, ext, -ext, ext])
plt.colorbar(im, ax=ax[0], fraction=0.046); ax[0].set_title("EXTRA - INTRA (rim difference)"); ax[0].set_xlabel("arcsec")
ax[1].plot(rax, pE / np.nanmax(pE), label="extra"); ax[1].plot(rax, pI / np.nanmax(pI), label="intra")
ax[1].set_xlabel("radius [arcsec]"); ax[1].set_ylabel("norm azimuthal mean"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title("intra vs extra radial profile")
fig.suptitle("FFT donut intra vs extra (on-axis): spherical breaks the symmetry at the rim", fontsize=12)
plt.show()

<a id='gridconv'></a>
## 7. FFT-grid convergence: 2048 vs 4096

The out-of-focus FFT needs a large pupil grid. Here we compare the detector-grid diffraction donut (extra and intra, broadband, no noise) at N=2048 vs N=4096 — the difference shows the residual dependence on the FFT grid.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
for row, dt in enumerate([DefocalType.Extra, DefocalType.Intra]):
    d2048 = diffraction_donut(dt, 0, N=2048, noise=False)
    d4096 = diffraction_donut(dt, 0, N=4096, noise=False)
    d2048 /= d2048.sum(); d4096 /= d4096.sum(); dd = d4096 - d2048
    v0 = max(d2048.max(), d4096.max()); vv = np.nanpercentile(np.abs(dd), 99.5)
    ax[row, 0].imshow(d2048, origin="lower", cmap=cmap, vmin=0, vmax=v0); ax[row, 0].set_title(f"{dt.value} N=2048")
    ax[row, 1].imshow(d4096, origin="lower", cmap=cmap, vmin=0, vmax=v0); ax[row, 1].set_title(f"{dt.value} N=4096")
    im = ax[row, 2].imshow(dd, origin="lower", cmap="RdBu_r", vmin=-vv, vmax=vv); plt.colorbar(im, ax=ax[row, 2], fraction=0.046)
    ax[row, 2].set_title(f"{dt.value} 4096 - 2048")
    for a in ax[row]: a.set_xticks([]); a.set_yticks([])
    print(f"{dt.value}: RMS(4096-2048) = {np.sqrt(np.mean(dd**2)):.2e} (unit-sum), "
          f"max |diff| / peak = {np.nanmax(np.abs(dd))/d4096.max()*100:.2f}%")
fig.suptitle("FFT-grid convergence: detector donut at N=2048 vs N=4096", fontsize=12)
plt.show()

<a id='fits'></a>
## 8. Danish Fits: Z11 intra/extra split (paired & unpaired)

Production danish config. A **paired** fit forces one wavefront (averaging the disagreement); **unpaired** fits expose it. We tabulate the spherical **Z11** for intra/extra and the fitted **fwhm** per side (geometric fits Z11 better but the fwhm match differs).

In [ ]:
results = {}
print(f"{'donut':<16s}{'Z11 E':>9s}{'Z11 I':>9s}{'Z11 I-E':>10s}{'Z11 pair':>10s}{'fwhm E':>9s}{'fwhm I':>9s}{'fwhm pair':>11s}{'resid':>8s}")
for kind in ["geometric", "FFT diffraction"]:
    zE, fE, hE = single_fit(donuts[kind]["extra"], DefocalType.Extra)
    zI, fI, hI = single_fit(donuts[kind]["intra"], DefocalType.Intra)
    zp, fp, hp = paired_fit(donuts[kind]["extra"], donuts[kind]["intra"])
    rE = np.std(hE["extra"]["image"] - hE["extra"]["model"])
    results[kind] = dict(zE=zE, zI=zI, fE=fE, fI=fI, zp=zp, fp=fp, hE=hE, hI=hI, resid=rE)
    print(f"{kind:<16s}{zE[i11]:>9.3f}{zI[i11]:>9.3f}{zI[i11]-zE[i11]:>10.3f}{zp[i11]:>10.3f}{fE:>9.2f}{fI:>9.2f}{fp:>11.2f}{rE:>8.0f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
kinds = ["geometric", "FFT diffraction"]; x = np.arange(len(kinds))
a1.bar(x, [1e3 * (results[k]["zI"][i11] - results[k]["zE"][i11]) for k in kinds], color=["C0", "C3"])
a1.axhline(300, color="k", ls=":", lw=1); a1.text(0, 300, " ~data (0.3 um)", fontsize=8, va="bottom")
a1.set_xticks(x); a1.set_xticklabels(kinds); a1.set_ylabel("Z11 intra - extra [nm]")
a1.set_title("Unpaired-fit spherical (Z11) intra/extra split"); a1.grid(alpha=0.3, axis="y")
d = results["FFT diffraction"]["zI"] - results["FFT diffraction"]["zE"]
a2.bar(range(len(noll)), 1e3 * d, color="C3"); a2.axvline(i11, color="k", ls=":", lw=1)
a2.set_xticks(range(len(noll))); a2.set_xticklabels([f"Z{j}" for j in noll], fontsize=7, rotation=90)
a2.set_ylabel("intra - extra [nm]"); a2.set_title("FFT donut: intra-extra by Zernike (Z11 marked)"); a2.grid(alpha=0.3, axis="y")
plt.show()

<a id='resid'></a>
## 9. Data / Model / Residual

Unpaired data / model / residual for the FFT (diffraction) donuts — the residual carries the diffraction structure the geometric model cannot represent, differing intra vs extra.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(11, 7.4), constrained_layout=True)
for row, side in enumerate(["extra", "intra"]):
    h = results["FFT diffraction"]["hE" if side == "extra" else "hI"]
    data = h[side]["image"]; model = h[side]["model"]; resid = data - model
    fw = results["FFT diffraction"]["fE" if side == "extra" else "fI"]
    for ax, arr, ttl, cm in [(axs[row, 0], data, "data", cmap), (axs[row, 1], model, "model", cmap),
                             (axs[row, 2], resid, "residual", "RdBu_r")]:
        if ttl == "residual":
            v = np.nanpercentile(np.abs(arr), 99.5); ax.imshow(arr, origin="lower", cmap=cm, vmin=-v, vmax=v)
        else:
            ax.imshow(arr, origin="lower", cmap=cm)
        ax.set_title(f"FFT {side} (fwhm {fw:.2f}) - {ttl}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.show()

<a id='summary'></a>
## 10. Summary

The cell below prints the conclusion from the **actual fitted numbers** (so it stays correct as the FFT grid / seeing change).

In [ ]:
g = results["geometric"]; d = results["FFT diffraction"]
g_split = 1e3 * (g["zI"][i11] - g["zE"][i11]); d_split = 1e3 * (d["zI"][i11] - d["zE"][i11])
g_pair = 1e3 * g["zp"][i11]; d_pair = 1e3 * d["zp"][i11]
print("=" * 72)
print(f"On-axis, FAM +/-{defocal_mm} mm, {band}-band, Kolmogorov {seeing_arcsec}\" seeing, N={Ngrid} grid,")
print(f"production danish config (21-term Noll, fwhm bound {fwhm_bounds}, unbinned):")
print("-" * 72)
print(f"  Z11 intra-extra split, UNPAIRED:  geometric {g_split:+.0f} nm   FFT-diffraction {d_split:+.0f} nm")
print(f"  Z11, PAIRED (one shared wavefront): geometric {g_pair:+.0f} nm   FFT-diffraction {d_pair:+.0f} nm")
print(f"  fitted fwhm (extra/intra):  geometric {g['fE']:.2f}/{g['fI']:.2f}\"   FFT {d['fE']:.2f}/{d['fI']:.2f}\"")
print(f"  fit residual RMS (extra):   geometric {g['resid']:.0f} e-   FFT {d['resid']:.0f} e-")
print("-" * 72)
print(f"=> Diffraction induces a {d_split:+.0f} nm Z11 intra/extra split in the SEPARATE (unpaired) fits")
print(f"   (vs {g_split:+.0f} nm geometric), AVERAGED AWAY in the paired fit ({d_pair:+.0f} nm).")
print(f"   Data shows ~+300 nm -> diffraction (Fraunhofer, on-axis) explains ~{abs(d_split)/300*100:.0f}% of it,")
print( "   with the right sign. Remaining: Fresnel near-field (Huygens), off-axis WFS spherical, 8mm giant defocus.")
print("=" * 72)

**Interpretation.** Diffraction — which danish's geometric fit neglects — produces a spherical
(Z11) intra/extra split in the *separately*-fit donuts with the same sign as the data, that is
**averaged away in the paired fit** (matching the observation that the disagreement appears only
when intra and extra are fit separately). Note the trade-off: the geometric donut recovers Z11
near-perfectly but the fitted fwhm differs from the diffraction case (the geometric model has no
diffraction-softened rim to absorb), while the diffraction donut pushes Z11 *and* fwhm. The full
data split (~0.3 um) likely also needs the pure **Fresnel near-field** term (Zemax-style Huygens,
`dx~lambda/D/20`, an overnight run), the **larger spherical at the off-axis WFS field positions**,
and the **8 mm giant-donut defocus**. The N=2048 vs 4096 test above bounds the FFT-grid systematic.
Struts are omitted (LSST's double-bladed spider is not easily modelled).